# ORCID Anthropologist Discovery

Created by [Matt Artz](https://www.mattartz.me/) — Advancing AI Anthropology through computational approaches to qualitative research.

---

## What This Notebook Does

This notebook searches the ORCID Public API to identify researchers who are likely anthropologists based on multiple signals in their ORCID profiles. The primary method uses education records (anthropology degrees), supplemented by employment data (anthropology departments), keywords, professional activities, and works metadata.

The workflow operates in two phases: first, a discovery phase that searches for potential anthropologists and scores them based on confidence signals; second, a data extraction phase that exports complete profile data for confirmed candidates in structured JSON format suitable for Wikidata import or other downstream processing.

## Key Features

- **Multi-Signal Detection**: Combines education (primary), employment, keywords, works, peer review, and professional activities to identify anthropologists
- **Confidence Scoring**: Assigns confidence levels based on signal strength and overlap
- **Subfield Detection**: Identifies anthropological subfields (cultural, biological, archaeological, linguistic, medical, applied)
- **Interactive Review**: Allows manual review of candidates before full data export
- **Comprehensive Export**: Outputs complete ORCID profile data in structured JSON
- **Wikidata-Ready**: Extracts data aligned with Wikidata person item properties

## Workflow

1. **Setup**: Configure API and search parameters
2. **Search**: Query ORCID for researchers matching anthropology criteria
3. **Score**: Evaluate candidates based on multiple signals
4. **Review**: Interactive review of candidates above confidence threshold
5. **Export**: Generate structured JSON for confirmed anthropologists

## Signal Hierarchy (Confidence Levels)

| Signal | Confidence | Notes |
|--------|------------|-------|
| PhD in Anthropology | High (0.9) | Primary identifier |
| MA/MS in Anthropology | High (0.8) | Primary identifier |
| BA/BS in Anthropology | Medium (0.5) | May have changed fields |
| Employment in Anthropology Dept | Medium (0.6) | Could be interdisciplinary |
| Keyword contains "anthropology" | Medium (0.5) | Self-identified |
| Publications in anthropology journals | Medium (0.5) | Active in field |
| Peer review for anthropology journals | Medium (0.4) | Field engagement |
| AAA membership | High (0.7) | Professional identity |

## Citation

If you use this notebook, please cite:

> Artz, Matt. (2026). MattArtzAnthro/wikidata-tools. Zenodo. https://doi.org/10.5281/zenodo.18912858

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

In [ ]:
# Install required packages
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import time
import re
import json
import os
from datetime import datetime
from typing import Dict, List, Optional, Set
from collections import defaultdict
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print("Setup complete.")

## Configuration

In [ ]:
class Config:
    ORCID_API_BASE = "https://pub.orcid.org/v3.0"
    ORCID_SEARCH_ENDPOINT = "https://pub.orcid.org/v3.0/search"
    REQUEST_DELAY = 0.5
    SEARCH_DELAY = 1.0
    TIMEOUT = 30
    MAX_RETRIES = 3
    SEARCH_PAGE_SIZE = 100
    MAX_SEARCH_RESULTS = 10000
    USER_AGENT = "ORCID-Anthropologist-Discovery/1.0 (Anthropology Research)"
    MINIMUM_CONFIDENCE = 0.4
    HIGH_CONFIDENCE = 0.7
    CLIENT_ID = None
    CLIENT_SECRET = None

config = Config()
print(f"ORCID API Base URL: {config.ORCID_API_BASE}")

## Anthropology Detection Patterns

In [ ]:
class AnthropologyPatterns:
    CORE_TERMS = [r'\banthropolog', r'\bethnograph', r'\bethnolog']
    
    SUBFIELDS = {
        'cultural': [r'cultural anthropolog', r'social anthropolog', r'sociocultural', r'ethnograph'],
        'biological': [r'biological anthropolog', r'physical anthropolog', r'primatolog', r'paleoanthropolog', r'forensic anthropolog'],
        'archaeological': [r'archaeolog', r'prehistor', r'bioarchaeolog'],
        'linguistic': [r'linguistic anthropolog', r'anthropological linguistic', r'ethnolinguist'],
        'medical': [r'medical anthropolog', r'health anthropolog'],
        'applied': [r'applied anthropolog', r'practicing anthropolog', r'design anthropolog', r'business anthropolog'],
        'visual': [r'visual anthropolog', r'ethnographic film'],
        'environmental': [r'environmental anthropolog', r'ecological anthropolog'],
    }
    
    DEGREE_PATTERNS = {
        'phd': [r'\bph\.?d\.?\b', r'\bdoctor', r'\bdoctoral'],
        'masters': [r'\bm\.?a\.?\b', r'\bm\.?s\.?\b', r'\bmaster'],
        'bachelors': [r'\bb\.?a\.?\b', r'\bb\.?s\.?\b', r'\bbachelor'],
    }
    
    DEPARTMENT_PATTERNS = [r'anthropolog', r'ethnolog', r'human sciences']
    
    PROFESSIONAL_ORGS = [
        r'american anthropological association', r'\baaa\b',
        r'society for american archaeology', r'\bsaa\b',
        r'european association of social anthropologists',
        r'society for applied anthropology', r'\bsfaa\b',
    ]
    
    JOURNALS = [
        r'american anthropologist', r'american ethnologist', r'cultural anthropology',
        r'current anthropology', r'annual review of anthropology',
        r'journal of the royal anthropological institute', r'ethnography',
        r'american journal of physical anthropology', r'journal of archaeological',
        r'medical anthropology', r'anthropological quarterly',
        r'human organization', r'practicing anthropology',
    ]

patterns = AnthropologyPatterns()
print(f"Loaded {len(patterns.SUBFIELDS)} subfields, {len(patterns.JOURNALS)} journal patterns")

## ORCID API Client

In [ ]:
class ORCIDClient:
    def __init__(self, config):
        self.config = config
        self.session = requests.Session()
        self.session.headers.update({'Accept': 'application/json', 'User-Agent': config.USER_AGENT})
        self._last_request_time = 0
    
    def _rate_limit(self, delay=None):
        delay = delay or self.config.REQUEST_DELAY
        elapsed = time.time() - self._last_request_time
        if elapsed < delay:
            time.sleep(delay - elapsed)
        self._last_request_time = time.time()
    
    def _make_request(self, url, params=None, delay=None):
        for attempt in range(self.config.MAX_RETRIES):
            self._rate_limit(delay)
            try:
                response = self.session.get(url, params=params, timeout=self.config.TIMEOUT)
                if response.status_code == 404:
                    return None
                response.raise_for_status()
                return response.json()
            except requests.exceptions.RequestException as e:
                if attempt < self.config.MAX_RETRIES - 1:
                    time.sleep(2 ** attempt)
                else:
                    print(f"Failed: {e}")
                    return None
        return None
    
    def search(self, query, start=0, rows=None):
        rows = rows or self.config.SEARCH_PAGE_SIZE
        return self._make_request(self.config.ORCID_SEARCH_ENDPOINT, 
                                  params={'q': query, 'start': start, 'rows': min(rows, 100)},
                                  delay=self.config.SEARCH_DELAY)
    
    def search_all(self, query, max_results=None, progress_callback=None):
        max_results = max_results or self.config.MAX_SEARCH_RESULTS
        orcids = []
        result = self.search(query, start=0)
        if not result:
            return []
        total = result.get('num-found', 0)
        print(f"Found {total} total results")
        
        for item in result.get('result', []):
            orcid_path = item.get('orcid-identifier', {}).get('path')
            if orcid_path:
                orcids.append(orcid_path)
        
        if progress_callback:
            progress_callback(len(orcids), min(total, max_results))
        
        while len(orcids) < total and len(orcids) < max_results:
            result = self.search(query, start=len(orcids))
            if not result:
                break
            batch = result.get('result', [])
            if not batch:
                break
            for item in batch:
                orcid_path = item.get('orcid-identifier', {}).get('path')
                if orcid_path:
                    orcids.append(orcid_path)
            if progress_callback:
                progress_callback(len(orcids), min(total, max_results))
        return orcids
    
    def get_record(self, orcid):
        return self._make_request(f"{self.config.ORCID_API_BASE}/{orcid}/record")
    
    def get_peer_reviews(self, orcid):
        return self._make_request(f"{self.config.ORCID_API_BASE}/{orcid}/peer-reviews")

client = ORCIDClient(config)
print("ORCID API client initialized.")

## Record Parsing Utilities

In [ ]:
def safe_get(data, *keys, default=None):
    current = data
    for key in keys:
        if isinstance(current, dict) and key in current:
            current = current[key]
        elif isinstance(current, list) and isinstance(key, int) and len(current) > key:
            current = current[key]
        else:
            return default
    return current

def parse_date(date_dict):
    if not date_dict:
        return None
    year = safe_get(date_dict, 'year', 'value')
    month = safe_get(date_dict, 'month', 'value')
    if not year:
        return None
    return f"{year}-{month.zfill(2)}" if month else year

def parse_affiliation(aff):
    org = safe_get(aff, 'organization', default={})
    return {
        'organization_name': safe_get(org, 'name'),
        'organization_city': safe_get(org, 'address', 'city'),
        'organization_country': safe_get(org, 'address', 'country'),
        'disambiguated_org_id': safe_get(org, 'disambiguated-organization', 'disambiguated-organization-identifier'),
        'disambiguated_org_source': safe_get(org, 'disambiguated-organization', 'disambiguation-source'),
        'role_title': safe_get(aff, 'role-title'),
        'department': safe_get(aff, 'department-name'),
        'start_date': parse_date(safe_get(aff, 'start-date')),
        'end_date': parse_date(safe_get(aff, 'end-date')),
    }

def extract_affiliations(activities_data, section, summary_key):
    items = []
    for group in safe_get(activities_data, section, 'affiliation-group', default=[]):
        for summary in safe_get(group, 'summaries', default=[]):
            item = safe_get(summary, summary_key)
            if item:
                items.append(parse_affiliation(item))
    return items

def extract_educations(activities_data):
    return extract_affiliations(activities_data, 'educations', 'education-summary')

def extract_employments(activities_data):
    return extract_affiliations(activities_data, 'employments', 'employment-summary')

def extract_memberships(activities_data):
    return extract_affiliations(activities_data, 'memberships', 'membership-summary')

def extract_services(activities_data):
    return extract_affiliations(activities_data, 'services', 'service-summary')

def extract_qualifications(activities_data):
    return extract_affiliations(activities_data, 'qualifications', 'qualification-summary')

def extract_distinctions(activities_data):
    return extract_affiliations(activities_data, 'distinctions', 'distinction-summary')

def extract_invited_positions(activities_data):
    return extract_affiliations(activities_data, 'invited-positions', 'invited-position-summary')

def extract_keywords(person_data):
    return [safe_get(kw, 'content') for kw in safe_get(person_data, 'keywords', 'keyword', default=[]) if safe_get(kw, 'content')]

def extract_works_summary(activities_data):
    works_groups = safe_get(activities_data, 'works', 'group', default=[])
    works_info = {'total_count': len(works_groups), 'by_type': defaultdict(int), 'journal_titles': set()}
    for group in works_groups:
        for summary in safe_get(group, 'work-summary', default=[]):
            works_info['by_type'][safe_get(summary, 'type', default='unknown')] += 1
            journal = safe_get(summary, 'journal-title', 'value')
            if journal:
                works_info['journal_titles'].add(journal)
            break
    works_info['by_type'] = dict(works_info['by_type'])
    works_info['journal_titles'] = list(works_info['journal_titles'])
    return works_info

def extract_peer_reviews(peer_review_data):
    reviews = []
    for group in safe_get(peer_review_data, 'group', default=[]):
        for review_group in safe_get(group, 'peer-review-group', default=[]):
            for summary in safe_get(review_group, 'peer-review-summary', default=[]):
                org = safe_get(summary, 'convening-organization', default={})
                reviews.append({'organization_name': safe_get(org, 'name'), 'role': safe_get(summary, 'role')})
    return reviews

print("Parsing functions defined.")

## Anthropologist Detection Functions

In [ ]:
def matches_pattern(text, pattern_list):
    if not text:
        return False
    text_lower = text.lower()
    return any(re.search(p, text_lower) for p in pattern_list)

def detect_anthropology_in_education(educations):
    result = {'is_anthropologist': False, 'confidence': 0.0, 'evidence': [], 'subfields': set(), 'highest_degree': None}
    degree_hierarchy = {'PhD': 3, 'Masters': 2, 'Bachelors': 1}
    
    for edu in educations:
        combined = f"{edu.get('role_title', '')} {edu.get('department', '')} {edu.get('organization_name', '')}".lower()
        if matches_pattern(combined, patterns.CORE_TERMS):
            result['is_anthropologist'] = True
            role = edu.get('role_title', '')
            degree_level, confidence = None, 0.5
            if matches_pattern(role, patterns.DEGREE_PATTERNS['phd']):
                degree_level, confidence = 'PhD', 0.9
            elif matches_pattern(role, patterns.DEGREE_PATTERNS['masters']):
                degree_level, confidence = 'Masters', 0.8
            elif matches_pattern(role, patterns.DEGREE_PATTERNS['bachelors']):
                degree_level, confidence = 'Bachelors', 0.5
            
            if degree_level and degree_hierarchy.get(degree_level, 0) > degree_hierarchy.get(result['highest_degree'], 0):
                result['highest_degree'] = degree_level
            result['confidence'] = max(result['confidence'], confidence)
            result['evidence'].append({'type': 'education', 'degree': degree_level, **edu})
            
            for sf, sp in patterns.SUBFIELDS.items():
                if matches_pattern(combined, sp):
                    result['subfields'].add(sf)
    result['subfields'] = list(result['subfields'])
    return result

def detect_anthropology_in_employment(employments):
    result = {'is_anthropologist': False, 'confidence': 0.0, 'evidence': [], 'subfields': set(), 'current_position': None}
    for emp in employments:
        combined = f"{emp.get('role_title', '')} {emp.get('department', '')} {emp.get('organization_name', '')}".lower()
        if matches_pattern(combined, patterns.DEPARTMENT_PATTERNS):
            result['is_anthropologist'] = True
            confidence = 0.6 if matches_pattern(emp.get('department', ''), patterns.DEPARTMENT_PATTERNS) else 0.4
            result['confidence'] = max(result['confidence'], confidence)
            if not emp.get('end_date'):
                result['current_position'] = emp
            result['evidence'].append({'type': 'employment', 'is_current': not emp.get('end_date'), **emp})
            for sf, sp in patterns.SUBFIELDS.items():
                if matches_pattern(combined, sp):
                    result['subfields'].add(sf)
    result['subfields'] = list(result['subfields'])
    return result

def detect_anthropology_in_keywords(keywords, biography=None):
    result = {'is_anthropologist': False, 'confidence': 0.0, 'evidence': [], 'subfields': set()}
    for kw in keywords:
        if matches_pattern(kw, patterns.CORE_TERMS):
            result['is_anthropologist'] = True
            result['confidence'] = max(result['confidence'], 0.5)
            result['evidence'].append({'type': 'keyword', 'keyword': kw})
            for sf, sp in patterns.SUBFIELDS.items():
                if matches_pattern(kw, sp):
                    result['subfields'].add(sf)
    if biography and matches_pattern(biography, patterns.CORE_TERMS):
        result['is_anthropologist'] = True
        result['confidence'] = max(result['confidence'], 0.4)
        result['evidence'].append({'type': 'biography', 'snippet': biography[:200]})
    result['subfields'] = list(result['subfields'])
    return result

def detect_anthropology_in_memberships(memberships):
    result = {'is_anthropologist': False, 'confidence': 0.0, 'evidence': []}
    for mem in memberships:
        org = f"{mem.get('organization_name', '')} {mem.get('department', '')}"
        if matches_pattern(org, patterns.PROFESSIONAL_ORGS):
            result['is_anthropologist'] = True
            result['confidence'] = max(result['confidence'], 0.7)
            result['evidence'].append({'type': 'membership', 'organization': mem.get('organization_name')})
    return result

def detect_anthropology_in_works(works_info):
    result = {'is_anthropologist': False, 'confidence': 0.0, 'evidence': [], 'anthro_journal_count': 0}
    for journal in works_info.get('journal_titles', []):
        if matches_pattern(journal, patterns.JOURNALS):
            result['is_anthropologist'] = True
            result['anthro_journal_count'] += 1
            result['evidence'].append({'type': 'publication', 'journal': journal})
    if result['anthro_journal_count'] >= 5:
        result['confidence'] = 0.6
    elif result['anthro_journal_count'] >= 2:
        result['confidence'] = 0.5
    elif result['anthro_journal_count'] >= 1:
        result['confidence'] = 0.4
    return result

def detect_anthropology_in_peer_reviews(peer_reviews):
    result = {'is_anthropologist': False, 'confidence': 0.0, 'evidence': []}
    for review in peer_reviews:
        org = review.get('organization_name', '')
        if matches_pattern(org, patterns.JOURNALS):
            result['is_anthropologist'] = True
            result['confidence'] = max(result['confidence'], 0.4)
            result['evidence'].append({'type': 'peer_review', 'journal': org})
    return result

print("Detection functions defined.")

## Candidate Evaluation

In [ ]:
def evaluate_candidate(orcid, record, peer_reviews=None):
    person_data = safe_get(record, 'person', default={})
    activities_data = safe_get(record, 'activities-summary', default={})
    
    name_data = safe_get(person_data, 'name', default={})
    given_name = safe_get(name_data, 'given-names', 'value') or ''
    family_name = safe_get(name_data, 'family-name', 'value') or ''
    full_name = f"{given_name} {family_name}".strip()
    biography = safe_get(person_data, 'biography', 'content')
    
    educations = extract_educations(activities_data)
    employments = extract_employments(activities_data)
    memberships = extract_memberships(activities_data)
    keywords = extract_keywords(person_data)
    works_info = extract_works_summary(activities_data)
    
    edu_result = detect_anthropology_in_education(educations)
    emp_result = detect_anthropology_in_employment(employments)
    kw_result = detect_anthropology_in_keywords(keywords, biography)
    mem_result = detect_anthropology_in_memberships(memberships)
    works_result = detect_anthropology_in_works(works_info)
    
    pr_list = extract_peer_reviews(peer_reviews) if peer_reviews else []
    pr_result = detect_anthropology_in_peer_reviews(pr_list)
    
    all_evidence = []
    all_subfields = set()
    for r in [edu_result, emp_result, kw_result, mem_result, works_result, pr_result]:
        all_evidence.extend(r.get('evidence', []))
        all_subfields.update(r.get('subfields', []))
    
    confidence_scores = {
        'education': edu_result['confidence'], 'employment': emp_result['confidence'],
        'keywords': kw_result['confidence'], 'membership': mem_result['confidence'],
        'works': works_result['confidence'], 'peer_review': pr_result['confidence'],
    }
    
    if edu_result['is_anthropologist']:
        base = edu_result['confidence']
        bonus = sum([0.03 if emp_result['is_anthropologist'] else 0,
                     0.02 if mem_result['is_anthropologist'] else 0,
                     0.01 if kw_result['is_anthropologist'] else 0,
                     0.02 if works_result['is_anthropologist'] else 0])
        final_confidence = min(base + bonus, 0.95)
        determination_basis = 'education'
    else:
        scores = [(emp_result['confidence'], 'employment'), (mem_result['confidence'], 'membership'),
                  (kw_result['confidence'], 'keywords'), (works_result['confidence'], 'works'),
                  (pr_result['confidence'], 'peer_review')]
        best_score, best_source = max(scores, key=lambda x: x[0])
        final_confidence = best_score * 0.8
        determination_basis = best_source
    
    is_anthro = any([edu_result['is_anthropologist'], emp_result['is_anthropologist'],
                     mem_result['is_anthropologist'], kw_result['is_anthropologist'],
                     works_result['is_anthropologist'], pr_result['is_anthropologist']])
    
    return {
        'orcid': orcid, 'orcid_url': f"https://orcid.org/{orcid}",
        'full_name': full_name, 'given_name': given_name, 'family_name': family_name,
        'is_anthropologist': is_anthro, 'confidence': round(final_confidence, 3),
        'confidence_level': 'high' if final_confidence >= config.HIGH_CONFIDENCE else 'medium' if final_confidence >= config.MINIMUM_CONFIDENCE else 'low',
        'determination_basis': determination_basis if is_anthro else None,
        'confidence_breakdown': confidence_scores, 'subfields': list(all_subfields),
        'evidence': all_evidence, 'highest_anthro_degree': edu_result.get('highest_degree'),
        'current_anthro_position': emp_result.get('current_position'),
        'education_count': len(educations), 'employment_count': len(employments),
        'works_count': works_info.get('total_count', 0), 'keyword_count': len(keywords),
    }

print("Evaluation function defined.")

## Search for Anthropologists

In [ ]:
search_queries = {
    'keyword_anthropology': 'keyword:anthropolog*',
    'biography_anthropology': 'biography:anthropolog*',
    'affiliation_anthropology': 'affiliation-org-name:anthropolog*',
    'cultural_anthro': 'keyword:"cultural anthropology"',
    'biological_anthro': '(keyword:"biological anthropology" OR keyword:"physical anthropology")',
    'archaeology': 'keyword:archaeolog*',
    'ethnography': '(keyword:ethnograph* OR biography:ethnograph*)',
}

print("Available search queries:")
for name, query in search_queries.items():
    print(f"  {name}: {query}")

In [ ]:
# Interactive search interface
query_dropdown = widgets.Dropdown(options=list(search_queries.items()), description='Search Type:')
custom_query = widgets.Text(placeholder='Or enter custom Solr query...', description='Custom:')
max_results_slider = widgets.IntSlider(value=500, min=100, max=5000, step=100, description='Max Results:')
search_button = widgets.Button(description='Search ORCID', button_style='primary')
search_progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:')
search_output = widgets.Output()

search_results = {'orcids': [], 'query': ''}

def on_search_click(b):
    global search_results
    with search_output:
        clear_output()
        query = custom_query.value.strip() or query_dropdown.value
        print(f"Searching: {query}")
        search_progress.value = 0
        def progress_cb(cur, tot):
            search_progress.max = tot
            search_progress.value = cur
        orcids = client.search_all(query, max_results=max_results_slider.value, progress_callback=progress_cb)
        search_results = {'orcids': orcids, 'query': query}
        print(f"Found {len(orcids)} ORCID records")

search_button.on_click(on_search_click)
display(widgets.VBox([query_dropdown, custom_query, max_results_slider, search_button, search_progress, search_output]))

## Evaluate Search Results

In [ ]:
eval_button = widgets.Button(description='Evaluate Candidates', button_style='success')
fetch_peer_reviews = widgets.Checkbox(value=False, description='Also fetch peer reviews (slower)')
eval_progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:')
eval_output = widgets.Output()

candidates = []

def on_eval_click(b):
    global candidates
    with eval_output:
        clear_output()
        orcids = search_results.get('orcids', [])
        if not orcids:
            print("No search results. Run search first.")
            return
        print(f"Evaluating {len(orcids)} records...")
        candidates = []
        eval_progress.max = len(orcids)
        for i, orcid in enumerate(orcids):
            try:
                record = client.get_record(orcid)
                if record:
                    pr = client.get_peer_reviews(orcid) if fetch_peer_reviews.value else None
                    ev = evaluate_candidate(orcid, record, peer_reviews=pr)
                    if ev['is_anthropologist'] and ev['confidence'] >= config.MINIMUM_CONFIDENCE:
                        candidates.append(ev)
            except Exception as e:
                pass
            eval_progress.value = i + 1
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(orcids)}, found {len(candidates)} candidates")
        candidates.sort(key=lambda x: x['confidence'], reverse=True)
        print(f"\nDone! Found {len(candidates)} anthropologists.")
        high = sum(1 for c in candidates if c['confidence'] >= config.HIGH_CONFIDENCE)
        print(f"  High confidence: {high}, Medium: {len(candidates) - high}")

eval_button.on_click(on_eval_click)
display(widgets.VBox([fetch_peer_reviews, eval_button, eval_progress, eval_output]))

## Review Candidates

In [ ]:
if candidates:
    df = pd.DataFrame(candidates)
    cols = ['orcid', 'full_name', 'confidence', 'confidence_level', 'determination_basis', 'highest_anthro_degree', 'subfields', 'works_count']
    print(f"High confidence candidates (>={config.HIGH_CONFIDENCE}):")
    display(df[df['confidence'] >= config.HIGH_CONFIDENCE][cols].head(30))
else:
    print("No candidates. Run search and evaluation first.")

## Export Full Profiles to JSON

In [ ]:
def extract_full_profile(orcid, record, peer_reviews=None):
    person = safe_get(record, 'person', default={})
    activities = safe_get(record, 'activities-summary', default={})
    name_data = safe_get(person, 'name', default={})
    
    other_names = [safe_get(n, 'content') for n in safe_get(person, 'other-names', 'other-name', default=[]) if safe_get(n, 'content')]
    emails = [safe_get(e, 'email') for e in safe_get(person, 'emails', 'email', default=[]) if safe_get(e, 'email')]
    urls = [{'name': safe_get(u, 'url-name'), 'url': safe_get(u, 'url', 'value')} for u in safe_get(person, 'researcher-urls', 'researcher-url', default=[])]
    countries = [safe_get(a, 'country', 'value') for a in safe_get(person, 'addresses', 'address', default=[]) if safe_get(a, 'country', 'value')]
    
    external_ids = {}
    for ext in safe_get(person, 'external-identifiers', 'external-identifier', default=[]):
        t, v = safe_get(ext, 'external-id-type'), safe_get(ext, 'external-id-value')
        if t and v:
            external_ids[t] = v
    
    fundings = []
    for g in safe_get(activities, 'fundings', 'group', default=[]):
        for s in safe_get(g, 'funding-summary', default=[]):
            org = safe_get(s, 'organization', default={})
            fundings.append({
                'title': safe_get(s, 'title', 'title', 'value'),
                'type': safe_get(s, 'type'),
                'organization': safe_get(org, 'name'),
                'start_date': parse_date(safe_get(s, 'start-date')),
                'end_date': parse_date(safe_get(s, 'end-date')),
            })
    
    return {
        'orcid': orcid, 'orcid_url': f"https://orcid.org/{orcid}", 'retrieved_at': datetime.now().isoformat(),
        'name': {
            'given_name': safe_get(name_data, 'given-names', 'value'),
            'family_name': safe_get(name_data, 'family-name', 'value'),
            'credit_name': safe_get(name_data, 'credit-name', 'value'),
            'full_name': f"{safe_get(name_data, 'given-names', 'value') or ''} {safe_get(name_data, 'family-name', 'value') or ''}".strip(),
            'other_names': other_names,
        },
        'biography': safe_get(person, 'biography', 'content'),
        'keywords': extract_keywords(person),
        'countries': countries, 'emails': emails, 'urls': urls, 'external_ids': external_ids,
        'educations': extract_educations(activities),
        'employments': extract_employments(activities),
        'memberships': extract_memberships(activities),
        'services': extract_services(activities),
        'qualifications': extract_qualifications(activities),
        'distinctions': extract_distinctions(activities),
        'invited_positions': extract_invited_positions(activities),
        'fundings': fundings,
        'works': extract_works_summary(activities),
        'peer_reviews': extract_peer_reviews(peer_reviews) if peer_reviews else [],
    }

print("Full profile extraction function defined.")

In [ ]:
# Export controls
export_source = widgets.RadioButtons(options=['All candidates', 'High confidence only'], value='High confidence only', description='Export:')
include_full_profile = widgets.Checkbox(value=True, description='Fetch full profile data (slower)')
include_peer_reviews_export = widgets.Checkbox(value=False, description='Include peer reviews')
export_button = widgets.Button(description='Export to JSON', button_style='warning')
export_progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:')
export_output = widgets.Output()

def on_export_click(b):
    with export_output:
        clear_output()
        to_export = candidates if export_source.value == 'All candidates' else [c for c in candidates if c['confidence'] >= config.HIGH_CONFIDENCE]
        if not to_export:
            print("No candidates to export.")
            return
        print(f"Exporting {len(to_export)} candidates...")
        
        export_data = {
            'metadata': {'exported_at': datetime.now().isoformat(), 'search_query': search_results.get('query', ''),
                         'total_candidates': len(to_export), 'full_profiles': include_full_profile.value},
            'candidates': []
        }
        export_progress.max = len(to_export)
        
        for i, cand in enumerate(to_export):
            if include_full_profile.value:
                try:
                    record = client.get_record(cand['orcid'])
                    pr = client.get_peer_reviews(cand['orcid']) if include_peer_reviews_export.value else None
                    if record:
                        profile = extract_full_profile(cand['orcid'], record, pr)
                        profile['anthropologist_evaluation'] = {
                            'confidence': cand['confidence'], 'confidence_level': cand['confidence_level'],
                            'determination_basis': cand['determination_basis'], 'subfields': cand['subfields'],
                            'highest_anthro_degree': cand['highest_anthro_degree'], 'evidence': cand['evidence'],
                        }
                        export_data['candidates'].append(profile)
                    else:
                        export_data['candidates'].append(cand)
                except:
                    export_data['candidates'].append(cand)
            else:
                export_data['candidates'].append(cand)
            export_progress.value = i + 1
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        json_file = f"orcid_anthropologists_{timestamp}.json"
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(export_data, f, indent=2, ensure_ascii=False, default=str)
        print(f"Saved: {json_file}")
        
        csv_file = f"orcid_anthropologists_summary_{timestamp}.csv"
        df_exp = pd.DataFrame([{
            'orcid': c.get('orcid'), 'full_name': c.get('name', {}).get('full_name') if isinstance(c.get('name'), dict) else c.get('full_name'),
            'confidence': c.get('anthropologist_evaluation', {}).get('confidence') or c.get('confidence'),
            'subfields': ', '.join(c.get('anthropologist_evaluation', {}).get('subfields') or c.get('subfields', [])),
            'highest_degree': c.get('anthropologist_evaluation', {}).get('highest_anthro_degree') or c.get('highest_anthro_degree'),
        } for c in export_data['candidates']])
        df_exp.to_csv(csv_file, index=False)
        print(f"Saved: {csv_file}")

export_button.on_click(on_export_click)
display(widgets.VBox([export_source, include_full_profile, include_peer_reviews_export, export_button, export_progress, export_output]))

In [ ]:
# Download files
import glob

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    for f in sorted(glob.glob('orcid_anthropologists_*.json'), reverse=True)[:1]:
        files.download(f)
    for f in sorted(glob.glob('orcid_anthropologists_*.csv'), reverse=True)[:1]:
        files.download(f)
else:
    print("Files saved to working directory:")
    for f in sorted(glob.glob('orcid_anthropologists_*.json'), reverse=True)[:1]:
        print(f"  {f}")
    for f in sorted(glob.glob('orcid_anthropologists_*.csv'), reverse=True)[:1]:
        print(f"  {f}")

## Manual ORCID Lookup

In [ ]:
orcid_input = widgets.Text(placeholder='0000-0002-1234-5678', description='ORCID:')
lookup_button = widgets.Button(description='Lookup & Evaluate', button_style='primary')
lookup_output = widgets.Output()

def on_lookup_click(b):
    with lookup_output:
        clear_output()
        orcid = orcid_input.value.strip().split('/')[-1]
        if not orcid:
            print("Enter an ORCID")
            return
        print(f"Looking up: {orcid}")
        record = client.get_record(orcid)
        if not record:
            print("Could not fetch record.")
            return
        pr = client.get_peer_reviews(orcid)
        ev = evaluate_candidate(orcid, record, peer_reviews=pr)
        
        print(f"\nName: {ev['full_name']}")
        print(f"URL: {ev['orcid_url']}")
        print(f"\nIs Anthropologist: {ev['is_anthropologist']}")
        print(f"Confidence: {ev['confidence']:.3f} ({ev['confidence_level']})")
        print(f"Basis: {ev['determination_basis']}")
        if ev['subfields']:
            print(f"Subfields: {', '.join(ev['subfields'])}")
        if ev['highest_anthro_degree']:
            print(f"Highest Degree: {ev['highest_anthro_degree']}")
        print(f"\nEducation: {ev['education_count']}, Employment: {ev['employment_count']}, Works: {ev['works_count']}")
        if ev['evidence']:
            print(f"\nEvidence ({len(ev['evidence'])} items):")
            for e in ev['evidence'][:5]:
                print(f"  - {e.get('type')}: {e.get('organization_name') or e.get('keyword') or e.get('journal') or e.get('organization')}")

lookup_button.on_click(on_lookup_click)
display(widgets.VBox([orcid_input, lookup_button, lookup_output]))

## JSON Output Schema

```json
{
  "metadata": {"exported_at": "...", "search_query": "...", "total_candidates": 123},
  "candidates": [{
    "orcid": "0000-0002-1234-5678",
    "name": {"given_name": "Jane", "family_name": "Smith", "full_name": "Jane Smith"},
    "biography": "...", "keywords": [...], "external_ids": {...},
    "educations": [...], "employments": [...], "works": {...},
    "anthropologist_evaluation": {
      "confidence": 0.92, "confidence_level": "high",
      "determination_basis": "education", "subfields": ["cultural"],
      "highest_anthro_degree": "PhD"
    }
  }]
}
```